# E0 Matched Baseline

展示型 notebook：训练 E0 workflow，运行 matched baseline 评估，并展示表格与图。

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate project root containing 'src'.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUTPUTS_ROOT = ROOT / "outputs"
print("Project root:", ROOT)
print("Outputs root:", OUTPUTS_ROOT)

import pandas as pd
from IPython.display import display

from src.config import setup_environment
from src.experiments import run_E0_evaluation, run_single_case_posterior
from src.plotting import (
    plot_e0_overall_figure,
    plot_feature_residuals,
    plot_observed_curve,
    plot_single_case_posterior,
    plot_single_case_ppc,
    plot_training_loss,
    plot_truth_vs_median_scatter,
)
from src.workflow import train_workflow

In [ ]:
missing = []
for pkg in ["bayesflow", "tensorflow", "keras"]:
    try:
        __import__(pkg)
        print(f"[ok] {pkg}")
    except Exception as e:
        missing.append(pkg)
        print(f"[missing] {pkg}: {e}")

if missing:
    print("\nInstall missing packages in the current kernel environment, for example:")
    print("python -m pip install bayesflow==2.0.7 tensorflow==2.17.0 keras==3.9.0")
else:
    print("\nBackend check passed.")

In [ ]:
# Optional install cell.
# Uncomment and run this cell if the backend packages are missing,
# then restart the kernel before continuing.

import sys
print(sys.executable)
# !{sys.executable} -m pip install bayesflow==2.0.7 tensorflow==2.17.0 keras==3.9.0 numpy==1.26.4 pandas==2.3.3 scipy==1.15.3 matplotlib==3.8.4 seaborn

In [ ]:
setup_environment(seed=42)
workflow_E0, history_E0 = train_workflow(seed=42)
plot_training_loss(history_E0)

In [ ]:
single_case = run_single_case_posterior(
    workflow=workflow_E0,
    true_beta=0.55,
    true_gamma=0.18,
    single_case_seed=2025,
    num_posterior_samples=4000,
    n_ppc=100,
    ppc_seed=2024,
)

plot_observed_curve(
    single_case["observed_curve"],
    single_case["time_grid"],
    true_beta=single_case["true_beta"],
    true_gamma=single_case["true_gamma"],
    true_r0=single_case["true_r0"],
)
display(single_case["summary_table"].round(4))
plot_single_case_posterior(
    single_case["posterior_df"],
    true_beta=single_case["true_beta"],
    true_gamma=single_case["true_gamma"],
    true_r0=single_case["true_r0"],
)
plot_single_case_ppc(
    single_case["ppc_curves"],
    single_case["observed_curve"],
    single_case["time_grid"],
)
print(f"True R0: {single_case['true_r0']:.3f}")
print(f"Posterior median R0: {single_case['r0_ci'][1]:.3f}")
print(f"Posterior 90% interval for R0: [{single_case['r0_ci'][0]:.3f}, {single_case['r0_ci'][2]:.3f}]")
print(f"Posterior probability R0 > 1: {single_case['prob_r0_gt_1']:.4f}")

In [ ]:
case_df, recovery_df, coverage_df, feature_case_df, feature_summary_df = run_E0_evaluation(
    workflow=workflow_E0,
    n_test=100,
    num_posterior_samples=2000,
    n_ppc_draws=100,
    seed=2026,
)

display(recovery_df.round(4))
display(coverage_df.round(4))
display(feature_summary_df.round(4))

In [ ]:
plot_truth_vs_median_scatter(case_df)
plot_feature_residuals(feature_case_df)
plot_e0_overall_figure(
    case_df,
    coverage_df,
    feature_summary_df,
    out_path=str(OUTPUTS_ROOT / "E0" / "E0_matched_overall_baseline_figure.png"),
)

In [ ]:
E0_OUTPUT_DIR = OUTPUTS_ROOT / "E0"
os.makedirs(E0_OUTPUT_DIR, exist_ok=True)
case_df.to_csv(E0_OUTPUT_DIR / "E0_matched_case_level.csv", index=False)
recovery_df.to_csv(E0_OUTPUT_DIR / "E0_matched_recovery_summary.csv", index=False)
coverage_df.to_csv(E0_OUTPUT_DIR / "E0_matched_coverage_summary.csv", index=False)
feature_case_df.to_csv(E0_OUTPUT_DIR / "E0_matched_ppc_feature_case_level.csv", index=False)
feature_summary_df.to_csv(E0_OUTPUT_DIR / "E0_matched_ppc_feature_summary.csv", index=False)
print("Saved E0 outputs.")